In [1]:
# Produce static color image with labels that you can share, view full screen, etc.
# Label the color image with id numbers or photo-z's
# like my old imlabel.py or bpzlabel.py

In [2]:
# color images
import PIL
from PIL import Image, ImageDraw, ImageFont
PIL.Image.MAX_IMAGE_PIXELS = 933120000  # allow to load large images avoiding DecompressionBombError

In [3]:
import astropy
from astropy.io import ascii
from astropy.table import Table
import astropy.units as u
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord

import numpy as np
import os

In [4]:
def between(lo, x, hi):
    return (lo < x) * (x < hi)

In [5]:
field = 'whl0137'

infile = 'WHL0137_full_MSA15.cat'

catalog = ascii.read(infile)
catalog[:2]

id,RA,Dec,redshift,fwhm,magnitude,magnitude_error,weight,reference,NRS_F110W,NRS_F140X,NRS_CLEAR
int64,float64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64
10000,24.346859,-8.464519,6.2,0.2,27.0,0.1,100000.0,0,27.0,27.0,27.0
10001,24.346627,-8.464261,6.2,0.2,27.0,0.1,30000.0,0,27.0,27.0,27.0


In [6]:
observed_targets = Table.read('MSAobs1.txt', format='ascii.tab')

In [7]:
observed_targets[:2]

id,weight,exposures,exp1,exp2,exp3
int64,int64,int64,str1,str1,str1
30604,5,3,x,x,x
30157,5,2,x,x,--


In [8]:
exposures = {}
n = len(observed_targets)
for i in range(n):
    id = observed_targets['id'][i]
    if id not in exposures.keys():
        exposures[id] = observed_targets['exposures'][i]
    else:
        exposures[id] += observed_targets['exposures'][i]
n

197

In [9]:
observed_targets = Table.read('MSAobs2.txt', format='ascii.tab')

In [10]:
n = len(observed_targets)
for i in range(n):
    id = observed_targets['id'][i]
    if id not in exposures.keys():
        exposures[id] = observed_targets['exposures'][i]
    else:
        exposures[id] += observed_targets['exposures'][i]
n

207

In [11]:
observed_targets = Table.read('MSAobs3.txt', format='ascii.tab')

In [12]:
n = len(observed_targets)
for i in range(n):
    id = observed_targets['id'][i]
    if id not in exposures.keys():
        exposures[id] = observed_targets['exposures'][i]
    else:
        exposures[id] += observed_targets['exposures'][i]       
n

197

In [13]:
exposures

{30604: 5,
 30157: 7,
 2178: 3,
 30066: 3,
 30698: 6,
 1153: 3,
 20111: 3,
 30273: 2,
 3122: 3,
 20161: 3,
 1005: 3,
 30077: 4,
 30533: 5,
 30619: 2,
 30577: 4,
 30310: 2,
 3167: 1,
 20201: 3,
 30332: 3,
 30150: 1,
 180: 3,
 30611: 3,
 30578: 6,
 12000: 3,
 30670: 5,
 504: 6,
 2776: 3,
 30047: 2,
 30336: 5,
 30214: 6,
 3117: 3,
 30465: 3,
 30706: 3,
 2534: 2,
 30049: 6,
 20216: 3,
 30439: 2,
 30817: 2,
 2251: 4,
 30532: 3,
 30128: 8,
 30441: 6,
 30335: 6,
 1004: 5,
 30708: 7,
 30819: 4,
 30200: 2,
 30763: 3,
 30301: 2,
 30333: 2,
 183: 4,
 2600: 3,
 2899: 6,
 30156: 5,
 3301: 3,
 30623: 8,
 30340: 2,
 30393: 4,
 30637: 1,
 30681: 3,
 30573: 2,
 2631: 3,
 30392: 2,
 893: 6,
 11000: 3,
 157: 4,
 3021: 3,
 30500: 5,
 2934: 3,
 30560: 9,
 30564: 2,
 1214: 3,
 30729: 1,
 30362: 6,
 2736: 5,
 30067: 3,
 10000: 6,
 10020: 3,
 30555: 5,
 30679: 3,
 30559: 3,
 30304: 3,
 30733: 1,
 30786: 3,
 30098: 3,
 1251: 1,
 3446: 2,
 969: 5,
 30664: 2,
 20295: 3,
 674: 2,
 30045: 1,
 30610: 6,
 30634: 5,


In [14]:
text_color = 255,255,0  # text color yellow because white blends into cores
text_font = ImageFont.load_default()  # text font

In [15]:
segm_dir = '../eazy_v4'
segm_file = 'sunrise-grizli-v4.0-ir_20mas_seg.fits.gz'
segm_file = os.path.join(segm_dir, segm_file)
segm_hdu = astropy.io.fits.open(segm_file)[0]
segm_data = segm_hdu.data
segm_wcs = WCS(segm_hdu.header)
segm_wcs

WCS Keywords

Number of WCS axes: 2
CTYPE : 'RA---TAN'  'DEC--TAN'  
CRVAL : 24.355  -8.457  
CRPIX : 7142.5  7886.5  
CD1_1 CD1_2  : -5.5555555555555e-06  0.0  
CD2_1 CD2_2  : 0.0  5.5555555555555e-06  
NAXIS : 17600  24000

In [16]:
catalog_coordinates = SkyCoord(ra=catalog['RA']*u.deg, dec=catalog['Dec']*u.deg)  # *u.deg
catalog['x'], catalog['y'] = segm_wcs.world_to_pixel(catalog_coordinates)

In [17]:
len(exposures)

455

In [18]:
science_targets = []
background_targets = []
n = len(catalog)
for i in range(0,n):
    id = catalog['id'][i]
    if id in exposures.keys():
        if between(19999, id, 50000):   # background
            background_targets.append(id)
        else:
            science_targets.append(id)

In [19]:
len(science_targets), len(background_targets)

(174, 281)

In [20]:
science_targets

[10000,
 10001,
 10002,
 10003,
 10020,
 10200,
 11000,
 11001,
 12001,
 12000,
 13001,
 13000,
 38,
 39,
 77,
 98,
 116,
 122,
 143,
 157,
 178,
 179,
 180,
 181,
 183,
 186,
 187,
 221,
 237,
 253,
 265,
 291,
 324,
 338,
 360,
 425,
 437,
 504,
 530,
 561,
 567,
 573,
 577,
 618,
 619,
 674,
 682,
 735,
 737,
 776,
 812,
 832,
 836,
 861,
 874,
 893,
 902,
 952,
 969,
 1003,
 1004,
 1005,
 1064,
 1148,
 1153,
 1164,
 1186,
 1214,
 1239,
 1243,
 1249,
 1251,
 1279,
 1291,
 1297,
 1299,
 1322,
 1348,
 1361,
 1363,
 1391,
 1393,
 1395,
 1429,
 1502,
 1547,
 1551,
 1686,
 1934,
 1968,
 2017,
 2090,
 2091,
 2110,
 2131,
 2147,
 2178,
 2209,
 2216,
 2221,
 2235,
 2236,
 2250,
 2251,
 2281,
 2326,
 2345,
 2367,
 2435,
 2440,
 2486,
 2534,
 2570,
 2574,
 2600,
 2608,
 2631,
 2633,
 2736,
 2745,
 2750,
 2771,
 2776,
 2808,
 2814,
 2833,
 2839,
 2872,
 2886,
 2899,
 2934,
 2936,
 2943,
 2952,
 3021,
 3037,
 3067,
 3078,
 3085,
 3100,
 3104,
 3114,
 3116,
 3117,
 3122,
 3167,
 3179,
 3205,
 32

In [21]:
exposure_catalog = Table()

In [22]:
exposure_catalog['id'] = science_targets

In [23]:
nexp = [exposures[id] for id in science_targets]

In [24]:
RAs = []
Decs = []
for id in science_targets:
    i = list(catalog['id']).index(id)
    RAs.append(catalog['RA'][i])
    Decs.append(catalog['Dec'][i])    

In [25]:
exposure_catalog['RA'] = RAs
exposure_catalog['Dec'] = Decs

In [26]:
exposure_catalog['exposures'] = nexp

In [27]:
exposure_catalog

id,RA,Dec,exposures
int64,float64,float64,int64
10000,24.346859,-8.464519,6
10001,24.346627,-8.464261,3
10002,24.34713,-8.464757,3
10003,24.3497,-8.466426,3
10020,24.347544,-8.465079,3
10200,24.348603,-8.465836,3
11000,24.346388,-8.464001,3
11001,24.346427,-8.464052,3
12001,24.34793,-8.465371,3


In [28]:
exposure_catalog['RA'].info.format = '%.6f'
exposure_catalog['Dec'].info.format = '%.6f'

In [29]:
exposure_catalog.write(field+'_MSA_exposures.cat', format='ascii.fixed_width_two_line', delimiter='  ') #, overwrite=True)

# Label color image

In [30]:
#color_image_file = '../color/whl0137_hst3.png'
color_image_file = '../color/whl0137_v4.png'
im = Image.open(color_image_file)
draw = ImageDraw.Draw(im)
nx, ny = im.size

In [31]:
n = len(catalog)
for i in range(n-1,-1,-1):
    id = catalog['id'][i]
    x = catalog['x'][i]
    y = ny - catalog['y'][i]
    if id in exposures.keys():
        if between(19999, id, 50000):   # background
            s = 'O'
            text_color = 255,0,255 # magenta
        else:
            s = '%d' % id
            text_color = 0,255,0  # green
        s += '[%d]' % exposures[id]
        print(s, x, y)
    else:
        if between(19999, id, 50000):
            s = None
        else:
            s = '%d' % id
            text_color = 0,0,255 # blue
    if s:
        draw.text((x, y), s, fill=text_color, font=text_font)

O[2] 12237.773101819213 8413.592516482175
O[2] 12759.680685513475 8413.632618607298
O[1] 13281.58829766524 8413.676627529578
O[3] 13803.495940913435 8413.724543254717
O[4] 11193.934382696889 8817.970430886391
O[4] 12237.743389669682 8818.03893249387
O[3] 12759.647930553532 8818.079045013641
O[2] 13281.552499891466 8818.12306534762
O[2] 14325.361734507176 8818.222829453003
O[1] 9628.20652147572 9222.343343828865
O[4] 10150.107916920831 9222.363922632121
O[4] 11193.910756173049 9222.416806696874
O[2] 11715.812205262097 9222.449111969783
O[1] 13281.51670211737 9222.569480724143
O[3] 13803.418259742613 9222.617421290848
O[3] 8062.497030467568 9626.751386027361
O[3] 10150.090376017124 9626.810263543686
O[6] 11715.785535926527 9626.895474954856
O[2] 12237.68396537447 9626.931698431443
O[1] 12759.582420632825 9626.971831746549
O[2] 13803.3794191597 9627.06382788739
O[2] 14325.27796771358 9627.115690716033
O[1] 5974.9105611281175 10031.20137253229
O[1] 6496.805838301343 10031.194586056936
O[1]

In [32]:
#im.show()
outfile = field + '_MSA_exposures.png'
im.save(outfile)